## Analysis 2: Photodegradability model of DPA under UV exposition

```=========================================================```

**NOTE:** This analysis wasn't able to be finished/completed due to computer limitations and on-going further reading of the psi4 documentation.

The results given on the current script are not yet reliable.

```=========================================================```

Before you proceed this workflow, I encourage you to follow the tutorials on ```00_tutorials/``` folder prior programming to have everything downloaded.

In this workflow, we will be modeling the photodegradability properties of DPA under UV exposition to evaluate how DPA is suitable for sunscreen.

Make sure you already have installed ```rdkit```, ```psi4```, ```pubchempy``` libraries. Otherwise, copy the following lines and paste them on the terminal:

1. ```conda activate vina```
2. ```conda install psi4 pubchempy rdkit```

## 1. Retrieve SMILES from DPA (and other sunscreen compounds)

PubChem CID of DPA = 10367

In [2]:
import pubchempy as pcp
from rdkit import Chem
from rdkit.Chem import AllChem

# Write PubChem CIDs from compounds
cids = [
    '51040', # Avobenzone
    '10367', # DPA
    # Write others
]

# Dictionary to store SMILES from CIDs
smiles_dict = {}
psi_conformers = {}
compound_names = {}

# Get the SMILES from compounds
for cid in cids:
    try:
        # 1. Retrieve SMILES from PubChem
        compound = pcp.Compound.from_cid(cid)
        smiles = compound.isomeric_smiles
        name = compound.iupac_name or compound.synonyms[0] if compound.synonyms else f"CID_{cid}"
        
        print(f"\n{'='*70}")
        print(f"CID {cid} - {name}")
        print(f"SMILES: {smiles}")
        print(f"{'='*70}")
        
        smiles_dict[cid] = smiles
        compound_names[cid] = name

        # 2. Convert SMILES to 3D structure with better parameters
        mol = Chem.MolFromSmiles(smiles)
        if mol is None:
            print(f"ERROR: Could not parse SMILES for CID {cid}")
            continue
            
        mol = Chem.AddHs(mol)
        
        # Try multiple conformers and pick the best one
        params = AllChem.ETKDG()
        params.randomSeed = 42
        params.useRandomCoords = True
        
        # Generate conformer
        confId = AllChem.EmbedMolecule(mol, params)
        
        if confId == -1:
            print(f"WARNING: Embedding failed, trying alternative method...")
            params.useRandomCoords = False
            confId = AllChem.EmbedMolecule(mol, params)
        
        if confId == -1:
            print(f"ERROR: Could not generate 3D structure for CID {cid}")
            continue
        
        # Optimize with MMFF
        converged = AllChem.MMFFOptimizeMolecule(mol, maxIters=500)
        print(f"MMFF optimization converged: {converged == 0}")
        
        # 3. Format for Psi4
        conf = mol.GetConformer()
        molecule_str = "0 1\n"
        for atom in mol.GetAtoms():
            pos = conf.GetAtomPosition(atom.GetIdx())
            molecule_str += f"{atom.GetSymbol()} {pos.x:10.6f} {pos.y:10.6f} {pos.z:10.6f}\n"
        
        print(f"3D Geometry generated successfully")
        print(f"Number of atoms: {mol.GetNumAtoms()}")
        psi_conformers[cid] = molecule_str
        
    except Exception as e:
        print(f"ERROR processing CID {cid}: {e}")
        continue

/tmp/ipykernel_24177/3225154095.py:22: PubChemPyDeprecationWarning: isomeric_smiles is deprecated: Use smiles instead
  smiles = compound.isomeric_smiles



CID 51040 - 1-(4-tert-butylphenyl)-3-(4-methoxyphenyl)propane-1,3-dione
SMILES: CC(C)(C)C1=CC=C(C=C1)C(=O)CC(=O)C2=CC=C(C=C2)OC
MMFF optimization converged: True
3D Geometry generated successfully
Number of atoms: 45

CID 10367 - pyridine-2,6-dicarboxylic acid
SMILES: C1=CC(=NC(=C1)C(=O)O)C(=O)O
MMFF optimization converged: True
3D Geometry generated successfully
Number of atoms: 17


# 2. Run TD-DFT Calculation

Calculate excited states (UV absorbtion).

In [3]:
import psi4
from psi4.driver.procrouting.response.scf_response import tdscf_excitations
import os

# Set memory and output file
psi4.set_memory('32 GB')
# Directory to store results
output_dir = 'output'
os.makedirs(output_dir, exist_ok=True)

# Store all results for final comparison
all_results = {}


  Memory set to  29.802 GiB by Python driver.


Run simulation per molecule

In [ ]:
# Run simulation per molecule
for cid in cids:
    if cid not in psi_conformers:
        print(f"\nSkipping CID {cid} - no valid geometry")
        continue

    print(f"\n{'='*70}")
    print(f"Processing CID: {cid} - {compound_names.get(cid, 'Unknown')}")
    print(f"{'='*70}\n")

    psi_mol_str = psi_conformers[cid]
    log_path = os.path.join(output_dir, f'cid_{cid}_tddft.dat')
    psi4.core.set_output_file(log_path, False)

    try:
        # Create Psi4 molecule
        mol = psi4.geometry(psi_mol_str)
        mol.update_geometry()

        # Set options for geometry optimization
        print(f"Optimizing geometry at B3LYP/6-31G* level...")
        psi4.set_options({
            'basis': '6-31G*',
            'reference': 'rks',
            'scf_type': 'df',
            'geom_maxiter': 50,
            'g_convergence': 'gau_tight'
        })
        
        # Optimize geometry with Psi4
        opt_energy = psi4.optimize('b3lyp', molecule=mol)
        print(f"Optimized geometry energy: {opt_energy:.8f} Hartree\n")

        # CRITICAL: Reset options with SAVE_JK for TD-DFT
        print(f"Running single-point B3LYP calculation for TD-DFT...")
        psi4.set_options({
            'basis': '6-31G*',
            'reference': 'rks',
            'scf_type': 'df',
            'save_jk': True  # REQUIRED for TD-DFT
        })
        
        # Run single point to get wavefunction with saved JK matrices
        energy, wfn = psi4.energy('b3lyp', molecule=mol, return_wfn=True)
        print(f"Ground state energy: {energy:.8f} Hartree\n")

        # Perform TD-DFT calculation
        print(f"Starting TD-DFT/TDA calculation...")
        n_states = 10  # Calculate more states
        res = tdscf_excitations(wfn, states=n_states, triplets='none', tda=True)
        
        # Extract and display results
        print(f"\nExcited States for CID {cid}:")
        print(f"{'State':<8} {'Energy (eV)':<15} {'Wavelength (nm)':<18} {'Osc. Strength':<15} {'Character':<15}")
        print("-" * 85)
        
        results = []
        for i, state_data in enumerate(res):
            state_num = i + 1
            
            ex_energy_ev = state_data['EXCITATION ENERGY']
            wavelength = 1239.84 / ex_energy_ev
            osc_strength = state_data['OSCILLATOR STRENGTH (LEN)']
            
            # Determine if this is a "bright" transition (high oscillator strength)
            character = "Bright" if osc_strength > 0.01 else "Dark"
            
            # Check if in UV range (280-400 nm)
            uv_region = ""
            if 280 <= wavelength <= 315:
                uv_region = "(UVB)"
            elif 315 < wavelength <= 400:
                uv_region = "(UVA)"
            
            print(f"{state_num:<8} {ex_energy_ev:<15.4f} {wavelength:<18.2f} {osc_strength:<15.6f} {character:<15} {uv_region}")
            
            results.append({
                'state': state_num,
                'energy_ev': ex_energy_ev,
                'wavelength_nm': wavelength,
                'osc_strength': osc_strength,
                'character': character
            })
            
            # Stop if wavelengths get too short (too high energy)
            if wavelength < 200:
                break
        
        all_results[cid] = {
            'name': compound_names.get(cid, f'CID_{cid}'),
            'states': results
        }

        print(f"\nDetailed log saved to '{log_path}'")
        
    except Exception as e:
        print(f"ERROR during calculation for CID {cid}: {e}")
        import traceback
        traceback.print_exc()
        continue

Some dependencies such as QCElemental have not yet finished migration to pydantic v2. If issues are encountered please downgrade pydantic or upgrade QCElemental as appropriate



Processing CID: 51040 - 1-(4-tert-butylphenyl)-3-(4-methoxyphenyl)propane-1,3-dione

Optimizing geometry at B3LYP/6-31G* level...


	Unable to completely converge to displaced geometry.
	RMS(dx):  7.179e-07 	Max(dx):  5.442e-06 	RMS(dq):  7.747e-05
	Previous geometry is closer to target in internal coordinates, so using that one.

	Best geometry has RMS(Delta(q)) = 7.75e-05

	Energy has increased in a minimization.
	Energy ratio indicates iffy step.
	Intrafrag trust radius decreased to   0.25.
	Unable to completely converge to displaced geometry.
	RMS(dx):  4.752e-07 	Max(dx):  1.328e-06 	RMS(dq):  1.151e-02
	Previous geometry is closer to target in internal coordinates, so using that one.

	Best geometry has RMS(Delta(q)) = 1.36e-02

	Energy has increased in a minimization.
	Energy ratio indicates iffy step.
	Intrafrag trust radius decreased to 0.0625.
	Previous geometry is closer to target in internal coordinates, so using that one.

	Best geometry has RMS(Delta(q)) = 3.37e-03

	Previous geometry is closer to target in internal coordinates, so using that one.

	Best geometry has RMS(Delta(q)) = 3.55e-03

	Previou

In [ ]:
# Final Summary
print(f"\n{'='*70}")
print("SUNSCREEN EFFECTIVENESS SUMMARY")
print(f"{'='*70}\n")

for cid, data in all_results.items():
    print(f"\n{data['name']} (CID: {cid}):")
    print("-" * 70)
    
    # Find strongest UV absorptions
    uv_absorptions = [s for s in data['states'] 
                      if 280 <= s['wavelength_nm'] <= 400 and s['osc_strength'] > 0.01]
    
    if uv_absorptions:
        print(f"UV-ACTIVE: Found {len(uv_absorptions)} bright transition(s) in UV range:")
        for state in uv_absorptions:
            region = "UVB" if state['wavelength_nm'] <= 315 else "UVA"
            print(f"\t - State {state['state']}: {state['wavelength_nm']:.1f} nm ({region}) - "
                  f"Oscillator strength: {state['osc_strength']:.4f}")
    else:
        print(f"NOT UV-ACTIVE: No bright absorptions in UV range (280-400 nm)")